# NOVA 2.0 — Train and test a small AI model

Run this notebook in Google Colab. Choose **Runtime → Change runtime type → T4 GPU** if available. This fine-tunes a small LoRA adapter on the reviewed examples in this repository. It is a starter experiment, not a complete self-learning system.

In [ ]:
!nvidia-smi || true
!pip -q install -U transformers==4.48.3 datasets peft accelerate bitsandbytes gradio

In [ ]:
import os, json, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from huggingface_hub import login

REPO = 'https://raw.githubusercontent.com/barmanbiswa131-design/My-app-bb/nova2-prototype/training/data/train.jsonl'
!wget -q -O /content/nova_train.jsonl "$REPO"
rows = [json.loads(x) for x in open('/content/nova_train.jsonl', encoding='utf-8') if x.strip()]
print('Training examples:', len(rows), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU')
BASE = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(BASE)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token


In [ ]:
def format_example(item):
    msgs = [
      {'role':'system','content':'You are NOVA, a friendly helpful assistant. Understand Bengali, Banglish, Hindi and Hinglish. Reply in the language used by the user. Be accurate and concise.'},
      {'role':'user','content':item['user']},
      {'role':'assistant','content':item['assistant']}
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False)

texts = [format_example(x) for x in rows]
def tokenize(batch):
    out = tokenizer(batch['text'], truncation=True, max_length=384)
    out['labels'] = [x[:] for x in out['input_ids']]
    return out
ds = Dataset.from_dict({'text':texts}).map(tokenize, batched=True, remove_columns=['text'])
print(ds)


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('GPU পাওয়া যায়নি। Colab Runtime → Change runtime type → T4 GPU নির্বাচন করে আবার চালাও।')
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=quant, device_map='auto')
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(task_type='CAUSAL_LM', r=8, lora_alpha=16, lora_dropout=0.05, target_modules=['q_proj','v_proj'], bias='none')
model = get_peft_model(model, lora)
model.print_trainable_parameters()
args = TrainingArguments(output_dir='/content/nova-checkpoints', num_train_epochs=3, per_device_train_batch_size=1, gradient_accumulation_steps=4, learning_rate=2e-4, logging_steps=1, save_strategy='epoch', fp16=True, report_to='none', remove_unused_columns=False)
trainer = Trainer(model=model, args=args, train_dataset=ds, data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False))
trainer.train()
ADAPTER='/content/nova-lora-adapter'
model.save_pretrained(ADAPTER); tokenizer.save_pretrained(ADAPTER)
print('Adapter saved:', ADAPTER)


In [ ]:
# Quick test of the newly trained adapter
def ask_nova(prompt):
    msgs=[{'role':'system','content':'You are NOVA. Reply naturally in Bengali, Banglish, Hindi or Hinglish, matching the user.'},{'role':'user','content':prompt}]
    rendered=tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs=tokenizer(rendered, return_tensors='pt').to(model.device)
    with torch.inference_mode():
        out=model.generate(**inputs, max_new_tokens=140, do_sample=True, temperature=0.7, top_p=0.9, repetition_penalty=1.08)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print(ask_nova('তোমার নাম কী?'))
print(ask_nova('Tum kaise ho?'))


## Open a temporary chat demo

This public Gradio link works only while this Colab session remains running. Do not enter private information.

In [ ]:
import gradio as gr
demo = gr.Interface(fn=ask_nova, inputs=gr.Textbox(lines=2, label='Message to NOVA'), outputs=gr.Textbox(label='NOVA reply'), title='NOVA 2.0 — Experimental AI')
demo.launch(share=True)